# 02 — Preprocessing and modelling

The full experiment is `python -m src.train` (it writes every result file). This notebook **reads those results** and shows the key steps, so the notebook can never disagree with the scripts. Run the training script first.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src import config
pd.set_option("display.max_columns", 50)

In [ ]:
R = config.RESULTS_DIR
load = lambda n: json.loads((R/n).read_text())
meta = json.loads(config.METADATA_PATH.read_text())
print('reportable:', meta.get('reportable'), '| final model:', meta['final_model'])

## Split

In [ ]:
split = load('split_info.json')
{k: split[k] for k in ['train_rows','test_rows','train_attack_share','test_attack_share','rare_classes_lt_threshold']}

## Feature-set comparison (cross-validation on the training subsample)

In [ ]:
fs = load('feature_selection.json')
print(fs['rule'], '->', fs['chosen'])
pd.DataFrame(fs['comparison'])

## Model comparison (test set; positive class = ATTACK)

In [ ]:
comp = pd.read_csv(R/'model_comparison.csv')
comp

In [ ]:
display(Image(str(config.FIGURES_DIR/'model_comparison.png')))
display(Image(str(config.FIGURES_DIR/'roc_curves.png')))
display(Image(str(config.FIGURES_DIR/'pr_curves.png')))

## Overfitting check: training-fold vs validation-fold F1

In [ ]:
comp[['model','cv_train_f1','cv_f1','overfit_gap_f1']]

## Error analysis

In [ ]:
err = load('error_analysis.json')
print('FN', err['false_negatives'], '| FP', err['false_positives'])
pd.DataFrame(err['detection_by_label'])

In [ ]:
display(Image(str(config.FIGURES_DIR/'detection_by_family.png')))
display(Image(str(config.FIGURES_DIR/'threshold_sweep.png')))

## Attack-family model and imbalance strategies

In [ ]:
mc = load('multiclass_results.json')
display(pd.DataFrame(mc['imbalance_comparison']))
pd.DataFrame({k: mc['test_report'][k] for k in mc['labels']}).T

## Held-out day

In [ ]:
ho = load('holdout_day_results.json')
print(ho['test_labels_unseen_in_training'])
pd.Series(ho['metrics'])

## Explainability

In [ ]:
ex = load('explainability.json')
pd.DataFrame(ex['permutation_importance']).head(15)

## Reload the saved pipeline and predict on the demo sample

In [ ]:
from src.predict import load_pipeline, load_schema, validate_frame, predict
demo = pd.read_csv(config.DEMO_SAMPLE_PATH)
v = validate_frame(demo, load_schema())
out = predict(load_pipeline(), v.X)
pd.crosstab(v.reference_labels, out['prediction'])